<a href="https://colab.research.google.com/github/pillaiganesh/MyAIAdventures/blob/HAAI%2B%2B/T5-Solve-Quiz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
This program is build with Flan-T5-XL LLM to be able to determine output of a MCQ question with four options.

> It accepts five parameters provided as a command line input.
> The first input represents the question and the next four input are the options.
> The output should be the option number: A/B/C/D
> Output should be upper-case
> There should be no additional output including any warning messages in the terminal.
> Remember that your output will be tested against test cases, therefore any deviation from the test cases will be considered incorrect during evaluation.


Syntax: python template.py <string> <string> <string> <string> <string>

The following example is given for your reference:

 Terminal Input: python template.py "What color is the sky on a clear, sunny day?" "Blue" "Green" "Red" "Yellow"
Terminal Output: A

 Terminal Input: python template.py "What color is the sky on a clear, sunny day?" "Green" "Blue" "Red" "Yellow"
Terminal Output: B

 Terminal Input: python template.py "What color is the sky on a clear, sunny day?" "Green" "Red" "Blue" "Yellow"
Terminal Output: C

 Terminal Input: python template.py "What color is the sky on a clear, sunny day?" "Green" "Red" "Yellow" "Blue"
Terminal Output: D

You are expected to create some examples of your own to test the correctness of your approach.

"""

import sys
import torch
import transformers
from transformers import T5Tokenizer, T5ForConditionalGeneration
import re

##### You may comment this section to see verbose -- but you must un-comment this before final submission. ######
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()
#################################################################################################################



def llm_function(model, tokenizer, q, a, b, c, d):
    '''
    The steps are given for your reference:

    1. Properly formulate the prompt as per the question - which should output either 'YES' or 'NO'. The output must always be upper-case. You may post-process to get the desired output.
    2. Tokenize the prompt
    3. Pass the tokenized prompt to the model get output in terms of logits since the output is deterministic.
    4. Extract the correct option from the model.
    5. Clean output and return.
    6. Output is case-sensative: A,B,C or D
    Note: The model (Flan-T5-XL) and tokenizer is already initialized. Do not modify that section.
    '''
    # Step 1: Create prompts for each option (YES/NO question)
    prompts = [
        f"Question: {q}\nAnswer: {a}\nIs this the correct answer? Answer YES or NO:",
        f"Question: {q}\nAnswer: {b}\nIs this the correct answer? Answer YES or NO:",
        f"Question: {q}\nAnswer: {c}\nIs this the correct answer? Answer YES or NO:",
        f"Question: {q}\nAnswer: {d}\nIs this the correct answer? Answer YES or NO:",
    ]

    # Step 2–4: Evaluate each option
    scores = []
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt")
        outputs = model.generate(**inputs, max_new_tokens=2)
        decoded = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().upper()
        scores.append(decoded.startswith("Y"))  # True if YES

    # Step 5: Pick the first option with YES
    mapping = {0: "A", 1: "B", 2: "C", 3: "D"}
    for idx, is_yes in enumerate(scores):
        if is_yes:
            return mapping[idx]

    # If none return YES, default to NA
    return "NA"

# Simulate command-line arguments in Colab
sys.argv = [
    "template.py",  # argv[0] is always the script name
    "What color is the sky on a clear, sunny day?",  # argv[1]
    "Blue",   # argv[2]
    "Green",  # argv[3]
    "Red",    # argv[4]
    "Yellow"  # argv[5]
]

if __name__ == '__main__':
    question = sys.argv[1].strip()
    option_a = sys.argv[2].strip()
    option_b = sys.argv[3].strip()
    option_c = sys.argv[4].strip()
    option_d = sys.argv[5].strip()

    ##################### Loading Model and Tokenizer ########################
    tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-xl")
    model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-xl")
    ##########################################################################

    """  Call to function that will perform the computation. """
    torch.manual_seed(42)
    out = llm_function(model,tokenizer,question,option_a,option_b,option_c,option_d)
    print(out.strip())

    """ End to call """